# Persian-GANwriting روی Google Colab (نسخه‌ی Closed-Vocabulary)

این نوت‌بوک نسخه‌ای از [Persian-GANwriting](https://github.com/mo0o0o0os/Persian-GANwriting)
(برنچ `claude/cool-hawking-cze7lx`) را روی Colab اجرا می‌کند.

**تغییر مهم نسبت به نسخه‌ی قبلی:** هدف از OOV واقعی (تولید کلمات کاملاً جدید)
به **style transfer با واژگان بسته** تغییر کرده — یعنی کلمه‌ی هدف هم همیشه
از همان ۱۲۵ کلمه‌ی دیتاست آبان انتخاب می‌شود، نه از یک پیکره‌ی خارجی. این کار
باعث می‌شود discriminator دیگر نتواند صرفاً با رد کردن «شکل‌های ناآشنا» تقلب
کند — یکی از دلایل اصلی mode collapse قبلی. جزئیات در `load_data.py` (متغیر
`CLOSED_VOCAB`) و `network_tro.py` (R1 penalty + label smoothing) هست.

**ترتیب اجرا:**
1. سلول‌های «راه‌اندازی» (کلون، نصب پکیج‌ها، فونت) را همیشه اجرا کنید.
2. اگر هنوز تصاویر دیتاست را ندارید یا فقط می‌خواهید مطمئن شوید کد سالم است
   → بخش **تست سریع (Smoke Test)** را اجرا کنید (چند ثانیه، بدون داده‌ی واقعی).
3. وقتی تصاویر واقعی آبان را در Drive گذاشتید → بخش **آموزش واقعی** را اجرا کنید.


## ۱) بررسی GPU
اگر خطا گرفتید: `Runtime → Change runtime type → T4 GPU` را انتخاب کنید و دوباره اجرا کنید.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
assert torch.cuda.is_available(), "GPU فعال نیست! از منوی Runtime -> Change runtime type -> GPU را انتخاب کنید."
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


## ۲) کلون کردن ریپو

اگر ریپو **پابلیک** است سلول زیر بدون تغییر کار می‌کند.
اگر **پرایوت** است، یک [Personal Access Token](https://github.com/settings/tokens)
با دسترسی `repo` بسازید و در پرامپتی که ظاهر می‌شود paste کنید (متن دیده نمی‌شود).


In [ ]:
import os

REPO_URL = "github.com/mo0o0o0os/Persian-GANwriting.git"
BRANCH = "claude/cool-hawking-cze7lx"

if not os.path.exists("/content/Persian-GANwriting"):
    is_private = False  # @param {type:"boolean"}
    if is_private:
        from getpass import getpass
        token = getpass("GitHub Personal Access Token: ")
        clone_url = f"https://{token}@{REPO_URL}"
    else:
        clone_url = f"https://{REPO_URL}"
    get_ipython().system(f"git clone -b {BRANCH} {clone_url} /content/Persian-GANwriting")
else:
    print("ریپو از قبل کلون شده؛ برای آپدیت: git -C /content/Persian-GANwriting pull")

os.chdir("/content/Persian-GANwriting")
print("CWD:", os.getcwd())


## ۳) نصب پکیج‌های موردنیاز
**توجه:** torch/torchvision را نصب نمی‌کنیم — همان نسخه‌ی از قبل نصب‌شده و سازگار با GPU خود Colab استفاده می‌شود (نسخه‌ی قدیمی `requirements.txt` قبلی برای Colab مناسب نبود).

In [ ]:
get_ipython().system("pip install -q python-Levenshtein albumentations arabic-reshaper python-bidi")
print("نصب شد.")


## ۴) دانلود فونت فارسی (برای پیش‌نمایش تصاویر آموزش در پوشه‌ی imgs/)

In [ ]:
import os
os.makedirs("fonts", exist_ok=True)
font_path = "fonts/Vazirmatn-Regular.ttf"
if not os.path.exists(font_path):
    get_ipython().system(
        "wget -q -O fonts/Vazirmatn-Regular.ttf "
        "https://github.com/rastikerdar/vazirmatn/raw/main/fonts/ttf/Vazirmatn-Regular.ttf"
    )
print("فونت آماده است." if os.path.exists(font_path) and os.path.getsize(font_path) > 0 else "دانلود فونت ناموفق بود؛ پیش‌نمایش‌ها fallback می‌شوند (روی آموزش اثر ندارد).")


## ۵) اتصال Google Drive (برای نگه‌داشتن دیتاست/چک‌پوینت‌ها بین سشن‌ها)

چون Colab رایگان معمولاً بعد از چند ساعت قطع می‌شود، دیتاست و پوشه‌های
`save_weights/` و `logs/` و `imgs/` را روی Drive می‌گذاریم تا با قطع شدن
سشن از دست نروند.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/Persian-GANwriting-run"  # دلخواه، عوضش کنید اگر خواستید
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("مسیر کار روی Drive:", DRIVE_ROOT)


## ۶) دیتاست واقعی آبان

تصاویر را (پوشه‌ی `words/` شامل فایل‌های png با همان نام‌هایی که در
`Groundtruth_farsi/*.filter27` آمده) از قبل در Drive، مثلاً در
`Persian-GANwriting-run/aban_words/`، قرار دهید. سلول زیر آن را به مسیری که
کد انتظار دارد (`./datasets/aban/words`) لینک می‌کند.

**اگر هنوز تصاویر واقعی را ندارید، این سلول را رد کنید و مستقیم به بخش
«تست سریع» بروید.**


In [ ]:
REAL_WORDS_DIR = f"{DRIVE_ROOT}/aban_words"  # پوشه‌ی حاوی png های واقعی روی Drive

os.makedirs("datasets/aban", exist_ok=True)
link_path = "datasets/aban/words"
if os.path.islink(link_path) or os.path.exists(link_path):
    get_ipython().system(f"rm -rf {link_path}")

if os.path.isdir(REAL_WORDS_DIR):
    os.symlink(REAL_WORDS_DIR, link_path)
    n = len(os.listdir(REAL_WORDS_DIR))
    print(f"لینک شد: {link_path} -> {REAL_WORDS_DIR} ({n} فایل)")
else:
    print(f"پوشه‌ی {REAL_WORDS_DIR} پیدا نشد -- تصاویر واقعی را اول در Drive آپلود کنید.")


---
## ۷) تست سریع (Smoke Test) -- اختیاری اما شدیداً پیشنهادی

بدون نیاز به تصاویر واقعی، یک دیتاست خیلی کوچک و مصنوعی (چند نویسنده،
تصاویر نویز تصادفی) با استفاده از فایل‌های **واقعی**
`Groundtruth_farsi` و `pairs_idx_wid_ABAN.py` می‌سازد و یک iteration کامل از
هر چهار مرحله‌ی آموزش (`rec_update`, `cla_update`, `dis_update`, `gen_update`)
را اجرا می‌کند. اگر این سلول بدون خطا تمام شد، یعنی pipeline (و تغییرات جدید
کد) روی این نسخه‌ی PyTorch سالم است و می‌توانید با خیال راحت آموزش واقعی و
طولانی را شروع کنید.


In [ ]:
import os, numpy as np, cv2

SMOKE_DIR = "/content/_smoketest"
SMOKE_IMG_DIR = f"{SMOKE_DIR}/words"
os.makedirs(SMOKE_IMG_DIR, exist_ok=True)

N_TRAIN_WRITERS = 6
N_TEST_WRITERS = 4

def make_fake_split(src_gt_path, writer_ids, out_gt_path):
    writer_ids = set(writer_ids)
    kept = []
    with open(src_gt_path, encoding="utf-8") as f:
        for line in f:
            wid = line.split(",")[0]
            if wid in writer_ids:
                kept.append(line.strip())
    with open(out_gt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(kept) + "\n")
    return kept

from pairs_idx_wid_ABAN import pairs_tr, pairs_te
tr_writer_ids = [wid for _, wid in pairs_tr[:N_TRAIN_WRITERS]]
te_writer_ids = [wid for _, wid in pairs_te[:N_TEST_WRITERS]]

tr_lines = make_fake_split("Groundtruth_farsi/gan.aban.tr_va.gt.filter27", tr_writer_ids, f"{SMOKE_DIR}/gt_tr.txt")
te_lines = make_fake_split("Groundtruth_farsi/gan.aban.test.gt.filter27", te_writer_ids, f"{SMOKE_DIR}/gt_te.txt")
print(f"{len(tr_lines)} خط train، {len(te_lines)} خط test برای تست سریع انتخاب شد.")

rng = np.random.default_rng(0)
for line in tr_lines + te_lines:
    idx = line.split(" ", 1)[0].split(",")[1]
    out_path = f"{SMOKE_IMG_DIR}/{idx}.png"
    if not os.path.exists(out_path):
        h, w = 64, rng.integers(40, 180)
        img = (rng.random((h, w)) * 255).astype("uint8")
        cv2.imwrite(out_path, img)
print("تصاویر نویزی ساختگی ساخته شدند:", len(os.listdir(SMOKE_IMG_DIR)))


In [ ]:
# مانکی‌پچ: load_data را موقتاً به دیتای ساختگی اشاره می‌دهیم (فایل‌های
# واقعی ریپو دست‌نخورده می‌مانند)
import importlib
import load_data
importlib.reload(load_data)
load_data.img_base = SMOKE_IMG_DIR
load_data.src = f"{SMOKE_DIR}/gt_tr.txt"
load_data.tar = f"{SMOKE_DIR}/gt_te.txt"

data_train, data_test = load_data.loadData(oov=True)
print("تعداد نویسنده‌های train/test در تست سریع:", len(data_train), len(data_test))

sample = data_train[list(data_train.data_dict.keys())[0]]
print("یک نمونه با موفقیت بارگذاری شد. شکل تصویر:", sample[3].shape)


In [ ]:
# یک batch واقعی از DataLoader + یک iteration کامل از هر ۴ مرحله‌ی آموزش
import torch
from torch import optim
from main_run import sort_batch
from network_tro import ConTranModel
from load_data import NUM_TRAIN_WRITERS
from loss_tro import CER

gpu = torch.device("cuda")
batch = [data_train[k] for k in list(data_train.data_dict.keys())[:4]]
batch = sort_batch(batch)

model = ConTranModel(NUM_TRAIN_WRITERS, 500, True).to(gpu)
dis_opt = optim.Adam(model.dis.parameters(), lr=8e-5)
gen_opt = optim.Adam(model.gen.parameters(), lr=8e-5)
rec_opt = optim.Adam(model.rec.parameters(), lr=8e-6)
cla_opt = optim.Adam(model.cla.parameters(), lr=8e-6)

cer = CER()
model.train()

rec_opt.zero_grad()
l_rec = model(batch, 0, "rec_update", cer)
rec_opt.step()
print("rec_update OK, loss =", l_rec.item())

cla_opt.zero_grad()
l_cla = model(batch, 0, "cla_update")
cla_opt.step()
print("cla_update OK, loss =", l_cla.item())

dis_opt.zero_grad()
l_dis = model(batch, 0, "dis_update")
dis_opt.step()
print("dis_update OK, loss =", l_dis.item())

gen_opt.zero_grad()
l_total, l_dis_g, l_cla_g, l_l1, l_rec_g = model(batch, 0, "gen_update", [CER(), CER()])
gen_opt.step()
print("gen_update OK, total loss =", l_total.item())

print("\n✅ تست سریع با موفقیت تمام شد -- pipeline سالم است.")


---
## ۸) آموزش واقعی

قبل از این بخش باید سلول «دیتاست واقعی آبان» (بخش ۶) با موفقیت اجرا شده
باشد. پوشه‌های `logs/`, `imgs/`, `save_weights/` را به Drive لینک می‌کنیم تا
با قطع شدن سشن از دست نروند، بعد `main_run.py` را طوری اجرا می‌کنیم که اگر
چک‌پوینتی از قبل روی Drive باشد، خودکار از همان‌جا ادامه بدهد.


In [ ]:
for name in ["logs", "imgs", "save_weights"]:
    drive_dir = f"{DRIVE_ROOT}/{name}"
    os.makedirs(drive_dir, exist_ok=True)
    local_path = name
    if os.path.islink(local_path):
        get_ipython().system(f"rm -f {local_path}")
    elif os.path.isdir(local_path):
        get_ipython().system(f"rm -rf {local_path}")
    os.symlink(drive_dir, local_path)
    print(f"{local_path} -> {drive_dir}")


In [ ]:
import glob, re

ckpts = glob.glob("save_weights/contran-*.model")
if ckpts:
    last_epoch = max(int(re.search(r"contran-(\d+)\.model", c).group(1)) for c in ckpts)
    print(f"چک‌پوینت پیدا شد -- ادامه از epoch {last_epoch}")
else:
    last_epoch = 0
    print("چک‌پوینتی پیدا نشد -- شروع از صفر")

# این سلول تا زمانی که Colab وصل است اجرا می‌ماند. اگر قطع شد، فقط دوباره
# این نوت‌بوک را از سلول ۱ اجرا کنید -- چون چک‌پوینت‌ها روی Drive هستند،
# همین سلول خودش از آخرین epoch ادامه می‌دهد.
get_ipython().system(f"python3 main_run.py {last_epoch}")


## ۹) مشاهده‌ی نتایج

- تصاویر نمونه هر ۵۰۰ iteration در `imgs/` (لینک‌شده به Drive) ذخیره می‌شوند.
- چک‌پوینت‌ها هر ۱۰۰ epoch در `save_weights/` ذخیره می‌شوند.
- لاگ کامل متنی در `logs/` هست؛ خط `l_discriminator=<train>-<gen>` را دنبال کنید:
  اگر مقدار train خیلی نزدیک صفر بماند ولی مقدار gen بالا/رو به رشد باشد،
  یعنی discriminator هنوز غالب است (`w_r1` در `network_tro.py` را بالاتر ببرید).
